In [1]:
import pandas as pd 
import numpy as np 


In [2]:
data=pd.read_excel("TRANSFORM DATA.xlsx")
data.head()

,Account Currency,Debit or Credit Account Number,Payment Method,Beneficiary Country/Jurisdiction Code,Beneficiary Zip Code,Branch Number,Branch Name,Beneficiary Account Type,Beneficiary Bank Routing Code,Transaction Reference Number,...,Country/Jurisdiction,Flow Name,End to End Id,Alias ID Type,Alias ID,Beneficiary Name,Additional Info,Unnamed: 102,Ultimate Remitter Name,Payment Details
0,UGX,100005003,DFT,NaN,NaN,886,KAMPALA CITIBANK,NaN,163047,3145686A,...,UG,NaN,NaN,,NaN,BALINDA ALEX,NaN,NaN,NaN,WITHDRAWAL EXEMPTED EMPLOYMENT BENE\nFIT NSSF ...
1,UGX,100005003,DFT,NaN,NaN,886,KAMPALA CITIBANK,NaN,163047,3145686A,...,UG,NaN,NaN,,NaN,BALINDA ALEX,NaN,NaN,NaN,WITHDRAWAL EXEMPTED EMPLOYMENT BENE\nFIT NSSF ...
2,UGX,100005003,DFT,NaN,NaN,886,KAMPALA CITIBANK,NaN,163047,3145686A,...,UG,FLOW FOR 100005003,NaN,,NaN,BALINDA ALEX,NaN,NaN,NaN,WITHDRAWAL EXEMPTED EMPLOYMENT BENE\nFIT NSSF ...
3,UGX,100005003,DFT,NaN,NaN,886,KAMPALA CITIBANK,NaN,163047,3145686A,...,UG,FLOW FOR 100005003,NaN,,NaN,BALINDA ALEX,NaN,NaN,NaN,WITHDRAWAL EXEMPTED EMPLOYMENT BENE\nFIT NSSF ...
4,UGX,100005003,DFT,NaN,NaN,886,KAMPALA CITIBANK,NaN,163047,3145686A,...,UG,FLOW FOR 100005003,NaN,,NaN,BALINDA ALEX,NaN,NaN,NaN,WITHDRAWAL EXEMPTED EMPLOYMENT BENE\nFIT NSSF ...


In [3]:
import pandas as pd

# Load the original file
df = pd.read_excel('TRANSFORM DATA.xlsx')

# Create full name
df['Full Name'] = (df['First Name'].fillna('') + ' ' + df['Last Name'].fillna('')).str.strip()

# Static columns (same for each transaction)
key_cols = [
    'Transaction Reference Number',
    'Beneficiary Name',
    'Payment Amount',
    'Payment Currency',
    'Payment Details',
    'Customer Number',
    'Customer Name',
    'Beneficiary Bank Name',
    'Branch Name',
    'Debit or Credit Account Number',
    'Payment Method',
    'Status',
    'Processing Date',
    'Value Date',
]

# One row of static info per transaction
static = df.groupby('Transaction Reference Number', as_index=False).first()[key_cols]

# VERIFY
verify = df[df['Action'] == 'VERIFY'][['Transaction Reference Number', 'Full Name', 'Date/Time Of Last Change']].copy()
verify = verify.rename(columns={'Full Name': 'Verifier', 'Date/Time Of Last Change': 'Verifier Date'})

# RELEASE
release = df[df['Action'] == 'RELEASE'][['Transaction Reference Number', 'Full Name', 'Date/Time Of Last Change']].copy()
release = release.rename(columns={'Full Name': 'Releaser', 'Date/Time Of Last Change': 'Release Date'})

# SUBMIT
submit = df[df['Action'] == 'SUBMIT'][['Transaction Reference Number', 'Full Name', 'Date/Time Of Last Change']].copy()
submit = submit.rename(columns={'Full Name': 'Submitter', 'Date/Time Of Last Change': 'Submit Date'})

# AUTHORIZE (can be multiple)
auth = df[df['Action'] == 'AUTHORIZE'][['Transaction Reference Number', 'Full Name', 'Date/Time Of Last Change']].copy()
auth_agg = auth.groupby('Transaction Reference Number').agg({
    'Full Name': lambda x: ' | '.join(x),
    'Date/Time Of Last Change': 'max'
}).reset_index()
auth_agg = auth_agg.rename(columns={'Full Name': 'Authorizers', 'Date/Time Of Last Change': 'Latest Authorize Date'})

# Merge everything
result = static.merge(verify, on='Transaction Reference Number', how='left')
result = result.merge(release, on='Transaction Reference Number', how='left')
result = result.merge(submit, on='Transaction Reference Number', how='left')
result = result.merge(auth_agg, on='Transaction Reference Number', how='left')

# Final column order
cols_order = [
    'Transaction Reference Number',
    'Beneficiary Name',
    'Payment Amount',
    'Payment Currency',
    'Payment Details',
    'Customer Number',
    'Customer Name',
    'Beneficiary Bank Name',
    'Branch Name',
    'Debit or Credit Account Number',
    'Payment Method',
    'Status',
    'Processing Date',
    'Value Date',
    'Submitter',
    'Submit Date',
    'Verifier',
    'Verifier Date',
    'Authorizers',
    'Latest Authorize Date',
    'Releaser',
    'Release Date',
]

result = result[cols_order].sort_values('Transaction Reference Number').reset_index(drop=True)

# Save
result.to_excel('Transformed_Payment_Data.xlsx', index=False)
print("Done!")
print(result)

Done!
  Transaction Reference Number Beneficiary Name  Payment Amount  \
0                     3145686A     BALINDA ALEX         2974331   
1                     3156580A    ANGUYO ANGELO         2476450   

  Payment Currency                                    Payment Details  \
0              UGX  WITHDRAWAL EXEMPTED EMPLOYMENT BENE\nFIT NSSF ...   
1              UGX  AGE 55 BENEFIT NSSF ACT/19631505014\n10/RCP174...   

   Customer Number                  Customer Name Beneficiary Bank Name  \
0           100005  NATIONAL SOCIAL SECURITY FUND        CENTENARY BANK   
1           100005  NATIONAL SOCIAL SECURITY FUND           EQUITY BANK   

        Branch Name  Debit or Credit Account Number  ... Processing Date  \
0  KAMPALA CITIBANK                       100005003  ...      2026-03-31   
1  KAMPALA CITIBANK                       100005003  ...      2026-04-10   

  Value Date            Submitter         Submit Date         Verifier  \
0 2026-04-01  NA 3 AUTOMATED FILE 2026-04-0

In [1]:
import pandas as pd

# Sample data
df = pd.DataFrame({
    'CustomerID': [101,101,101,101,102,102,102,102],
    'CustomerName': ['Alice','Alice','Alice','Alice','Bob','Bob','Bob','Bob'],
    'ProcessStep': ['submit','verify','authorise','release','submit','verify','authorise','release'],
    'StepOwner': ['Alice','Joe','Sarah','Tom','Bob','Joe','Lisa','Lisa'],
    'StepDate': pd.to_datetime(['2025-01-01','2025-01-02','2025-01-03','2025-01-04',
                                '2025-01-05','2025-01-06','2025-01-07','2025-01-08'])
})

# Pivot for names
names = df.pivot_table(index=['CustomerID','CustomerName'], 
                       columns='ProcessStep', 
                       values='StepOwner', 
                       aggfunc='first').reset_index()

# Pivot for dates
dates = df.pivot_table(index=['CustomerID','CustomerName'], 
                       columns='ProcessStep', 
                       values='StepDate', 
                       aggfunc='first').reset_index()

# Merge and rename
result = names.merge(dates, on=['CustomerID','CustomerName'], suffixes=('_name','_date'))
result.rename(columns={
    'verify_name': 'VerifierName',
    'authorise_name': 'AuthoriserName',
    'verify_date': 'VerifyDate',
    'authorise_date': 'AuthoriseDate'
}, inplace=True)

# Keep only needed columns
result = result[['CustomerID','CustomerName','VerifierName','VerifyDate','AuthoriserName','AuthoriseDate']]
print(result)

ProcessStep  CustomerID CustomerName VerifierName VerifyDate AuthoriserName  \
0                   101        Alice          Joe 2025-01-02          Sarah   
1                   102          Bob          Joe 2025-01-06           Lisa   

ProcessStep AuthoriseDate  
0              2025-01-03  
1              2025-01-07  


Perfect! Let me show you **exactly how to do this in Excel** with step-by-step instructions, including screenshots in text form.

---

## Method 1: Power Query (Best for assignments & repeatable work)

This is the **professional approach** that Excel recommends for this type of problem.

### Step-by-step instructions:

**Step 1: Select your data**
- Click anywhere inside your data table
- Press `Ctrl + T` to convert to an Excel Table (optional but recommended)
- Name your table (Table Design tab → Table Name: "Transactions")

**Step 2: Open Power Query**
- Go to **Data tab** → **Get & Transform Data** group
- Click **From Table/Range** (if you made a table)
- Or **From Sheet** if just selecting range

**Step 3: Pivot the ProcessStep column for names**
- In Power Query Editor, select the **ProcessStep** column
- Go to **Transform tab** → **Pivot Column**
- In the dialog:
  - **Values Column:** Select `StepOwner`
  - **Advanced Options:** Click dropdown → Select `"Don't Aggregate"`
  - Click **OK**

**Step 4: Pivot again for dates**
- Now select the **ProcessStep** column again
- **Transform tab** → **Pivot Column** again
- In the dialog:
  - **Values Column:** Select `StepDate`
  - **Advanced Options:** `"Don't Aggregate"`
  - Click **OK**

**Step 5: Merge the two pivoted tables**
- You'll now have two separate queries (Power Query creates them separately)
- Go to **Home tab** → **Merge Queries** → **Merge Queries as New**
- Select first query and second query
- Join on: `CustomerID` and `CustomerName` (hold Ctrl to select multiple)
- Join Kind: `Left Outer`

**Step 6: Expand and clean up**
- Click the expand icon (double arrows) on the new column
- Uncheck `CustomerID` and `CustomerName` (already have them)
- Keep only: `verify`, `authorise` (from first pivot) and `verify`, `authorise` (from second pivot)

**Step 7: Rename columns**
- Double-click column headers to rename:
  - `verify` (from StepOwner) → `VerifierName`
  - `authorise` (from StepOwner) → `AuthoriserName`
  - `verify` (from StepDate) → `VerifyDate`
  - `authorise` (from StepDate) → `AuthoriseDate`

**Step 8: Remove unnecessary columns**
- Remove `submit` and `release` columns (if you don't need them)
- Remove any other extra columns

**Step 9: Load back to Excel**
- **Home tab** → **Close & Load** → **Close & Load To...**
- Choose `New Worksheet` or `Existing Worksheet`

---

## Method 2: Excel Formulas (No Power Query)

Use this if you can't use Power Query for some reason.

### Setup your data:
Assume your raw data is in columns A-E:
- Column A: CustomerID
- Column B: CustomerName
- Column C: ProcessStep
- Column D: StepOwner
- Column E: StepDate

### Step 1: Create unique customer list
In cell G2 (or any empty column), enter:
```excel
=UNIQUE(A2:B100)
```
This gives you each customer once.

### Step 2: Get VerifierName
In cell I2 (next to your unique list), enter:
```excel
=XLOOKUP(1, (A:A=G2)*(C:C="verify"), D:D, "Not found")
```
Or using FILTER:
```excel
=FILTER(D:D, (A:A=G2)*(C:C="verify"), "Not found")
```

### Step 3: Get VerifyDate
In cell J2:
```excel
=XLOOKUP(1, (A:A=G2)*(C:C="verify"), E:E, "Not found")
```

### Step 4: Get AuthoriserName
In cell K2:
```excel
=XLOOKUP(1, (A:A=G2)*(C:C="authorise"), D:D, "Not found")
```

### Step 5: Get AuthoriseDate
In cell L2:
```excel
=XLOOKUP(1, (A:A=G2)*(C:C="authorise"), E:E, "Not found")
```

### Step 6: Drag down
Drag all formulas down for each unique customer.

---

## Method 3: PivotTable (Quick but less flexible)

**Step 1:** Insert PivotTable
- Select your data → **Insert tab** → **PivotTable**

**Step 2:** Build the PivotTable
- Rows: `CustomerID`, `CustomerName`
- Columns: `ProcessStep`
- Values: `StepOwner` (for names) and `StepDate` (for dates)

**Problem:** PivotTable will show counts, not the actual names/dates.

**Solution:** Use "Show values as" → but this is messy. **Not recommended** for this use case.

---

## Method 4: Simple helper columns (Beginner friendly)

This is the most manual but easiest to understand.

### Step 1: Create helper columns
Add columns to your raw data:
- Column F: `=C2&"_"&A2` (e.g., "verify_101")
- Column G: `=C2&"_Date_"&A2` (e.g., "verify_Date_101")

### Step 2: Create unique customer list
Copy columns A:B → **Data tab** → **Remove Duplicates**

### Step 3: Get verifier using INDEX-MATCH
For CustomerID in cell H2:
```excel
=INDEX(D:D, MATCH("verify_"&H2, F:F, 0))
```

### Step 4: Get verify date
```excel
=INDEX(E:E, MATCH("verify_Date_"&H2, G:G, 0))
```

### Step 5: Get authoriser
```excel
=INDEX(D:D, MATCH("authorise_"&H2, F:F, 0))
```

### Step 6: Get authorise date
```excel
=INDEX(E:E, MATCH("authorise_Date_"&H2, G:G, 0))
```

---

## My recommendation for your assignment

**Use Method 1 (Power Query)** because:
1. ✅ Handles different verifiers automatically
2. ✅ One-time setup, then refreshes with one click
3. ✅ Professional technique (good for research work)
4. ✅ No formulas to drag or maintain
5. ✅ Works for thousands of rows

---

## Quick demo with example data

Let me show you what happens with different verifiers:

**Raw data (3 customers, different verifiers):**

| CustomerID | CustomerName | ProcessStep | StepOwner | StepDate   |
|------------|--------------|-------------|-----------|------------|
| 101        | Alice        | verify      | **Joe**   | 2025-01-02 |
| 102        | Bob          | verify      | **Maria** | 2025-01-06 |
| 103        | Charlie      | verify      | **Joe**   | 2025-01-10 |

**After Power Query pivot:**

| CustomerID | CustomerName | VerifierName | VerifyDate |
|------------|--------------|--------------|------------|
| 101        | Alice        | Joe          | 2025-01-02 |
| 102        | Bob          | Maria        | 2025-01-06 |
| 103        | Charlie      | Joe          | 2025-01-10 |

Notice:
- Each customer gets THEIR verifier
- Joe appears twice (verified Alice AND Charlie) - this is correct!

---

## Troubleshooting common issues

**Issue 1:** Power Query says "We cannot convert a value to type Date"
- **Fix:** Select StepDate column → Transform tab → Data Type → Date

**Issue 2:** Getting multiple columns like verify.1, verify.2
- **Fix:** Before pivoting, sort by StepDate, then use "Keep Rows" → "Keep Duplicates" to keep only latest

**Issue 3:** #N/A errors with XLOOKUP
- **Fix:** Wrap in IFERROR: `=IFERROR(XLOOKUP(...), "No verify found")`

**Issue 4:** Customer appears multiple times in output
- **Fix:** Make sure you're using UNIQUE or removing duplicates before applying formulas

---

Would you like me to create a **step-by-step video script** or **screenshot guide** you can follow exactly? Or explain how to handle it if your data is in a different structure (like all in one column)?